# Baseline: taxa de conversão da regra fixa

In [3]:
import pandas as pd

y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()
arm_test = pd.read_csv("../data/processed/arm_test.csv").squeeze()

# Baseline: sempre recomenda o braço 0 (cellular).
# A métrica de conversão do baseline é a taxa de conversão real observada
# no braço 0 dentro do conjunto de teste — é o que aconteceria se a regra
# fixa fosse aplicada a todo mundo.
baseline_arm = 0
mask_baseline = arm_test == baseline_arm
baseline_conversion = y_test[mask_baseline].mean()

print(f"Baseline (sempre braço {baseline_arm} / cellular)")
print(f"Conversão observada no braço {baseline_arm} (teste): {baseline_conversion:.4f} ({baseline_conversion*100:.2f}%)")
print(f"Tamanho da amostra (braço {baseline_arm} no teste): {mask_baseline.sum()}")

Baseline (sempre braço 0 / cellular)
Conversão observada no braço 0 (teste): 0.1488 (14.88%)
Tamanho da amostra (braço 0 no teste): 5236


Na base de teste o contato via celular teve rendimento de 14,88%

# Treinamento de regressão logistica
## Modelo para cada braço (0-celular/1-telefone)

In [4]:
from sklearn.linear_model import LogisticRegression

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
arm_train = pd.read_csv("../data/processed/arm_train.csv").squeeze()

modelos_por_braco = {}

for braco in [0, 1]:
    mask = arm_train == braco
    modelo = LogisticRegression(max_iter=1000, random_state=42)
    modelo.fit(X_train[mask], y_train[mask])
    modelos_por_braco[braco] = modelo
    print(f"Braço {braco}: treinado com {mask.sum()} linhas")

/home/dev/miniconda3/envs/tc5/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Braço 0: treinado com 20908 linhas
Braço 1: treinado com 12042 linhas


/home/dev/miniconda3/envs/tc5/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
import numpy as np

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()
arm_test = pd.read_csv("../data/processed/arm_test.csv").squeeze()

np.random.seed(42)
epsilon = 0.10

# Probabilidade de conversão prevista por cada modelo, para todo o conjunto de teste
prob_braco_0 = modelos_por_braco[0].predict_proba(X_test)[:, 1]
prob_braco_1 = modelos_por_braco[1].predict_proba(X_test)[:, 1]

n = len(X_test)
escolhas = np.empty(n, dtype=int)

for i in range(n):
    if np.random.rand() < epsilon:
        escolhas[i] = np.random.choice([0, 1])  # exploração: braço aleatório
    else:
        escolhas[i] = 0 if prob_braco_0[i] >= prob_braco_1[i] else 1  # aproveitamento: melhor previsto

# Avaliação por "replay": só contamos os casos em que a escolha do bandit
# bateu com o braço que realmente aconteceu no histórico (arm_test) —
# é a única forma honesta de estimar conversão sem dado sintético.
mask_match = escolhas == arm_test.values
conversao_bandit = y_test[mask_match].mean()
cobertura = mask_match.mean()

print(f"Epsilon-Greedy (epsilon={epsilon})")
print(f"Distribuição de escolhas do bandit: braço 0 = {(escolhas==0).sum()}, braço 1 = {(escolhas==1).sum()}")
print(f"Casos onde a escolha bateu com o histórico (replay): {mask_match.sum()} de {n} ({cobertura*100:.2f}%)")
print(f"Conversão estimada do Epsilon-Greedy (replay): {conversao_bandit:.4f} ({conversao_bandit*100:.2f}%)")

Epsilon-Greedy (epsilon=0.1)
Distribuição de escolhas do bandit: braço 0 = 6382, braço 1 = 1856
Casos onde a escolha bateu com o histórico (replay): 4466 de 8238 (54.21%)
Conversão estimada do Epsilon-Greedy (replay): 0.1397 (13.97%)


In [6]:
resultados_epsilon = []

for epsilon in [0.10, 0.05, 0.01]:
    np.random.seed(42)
    escolhas_eps = np.empty(n, dtype=int)
    for i in range(n):
        if np.random.rand() < epsilon:
            escolhas_eps[i] = np.random.choice([0, 1])
        else:
            escolhas_eps[i] = 0 if prob_braco_0[i] >= prob_braco_1[i] else 1
    mask = escolhas_eps == arm_test.values
    conversao = y_test[mask].mean()
    cobertura = mask.mean()
    resultados_epsilon.append({
        "politica": f"Epsilon-Greedy (epsilon={epsilon})",
        "cobertura_replay": round(cobertura, 4),
        "conversao_estimada": round(conversao, 4),
    })

tabela_comparativa = pd.DataFrame(
    [{"politica": "Baseline (sempre cellular)", "cobertura_replay": 1.0, "conversao_estimada": round(baseline_conversion, 4)}]
    + resultados_epsilon
)
tabela_comparativa["conversao_%"] = (tabela_comparativa["conversao_estimada"] * 100).round(2)
tabela_comparativa["lift_vs_baseline_pp"] = ((tabela_comparativa["conversao_estimada"] - baseline_conversion) * 100).round(2)

tabela_comparativa

,politica,cobertura_replay,conversao_estimada,conversao_%,lift_vs_baseline_pp
0,Baseline (sempre cellular),1.0000,0.1488,14.88,0.00
1,Epsilon-Greedy (epsilon=0.1),0.5421,0.1397,13.97,-0.91
2,Epsilon-Greedy (epsilon=0.05),0.5426,0.1418,14.18,-0.70
3,Epsilon-Greedy (epsilon=0.01),0.5460,0.1421,14.21,-0.67


o baseline determinístico (sempre recomendar cellular) atingiu 14,88% de conversão no teste. O Epsilon-Greedy contextual (modelo de reward por braço via regressão logística) não superou o baseline em nenhum epsilon testado (10%, 5%, 1%), ficando entre 13,97% e 14,21% — mas convergindo para o baseline conforme epsilon diminui, como esperado. Hipóteses para a diferença: (1) o modelo do braço telephone, treinado com menos exemplos positivos, pode estar mal calibrado; (2) a avaliação por replay tem cobertura parcial (~54%) e pode ser enviesada, já que a atribuição histórica de canal não foi aleatória. Fica registrado como próximo ajuste possível (fora do escopo mínimo desta etapa): treinar os modelos com class_weight="balanced".

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

SEED = 42

# Reload dos dados (célula independente das anteriores)
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()
arm_train = pd.read_csv("../data/processed/arm_train.csv").squeeze()
arm_test = pd.read_csv("../data/processed/arm_test.csv").squeeze()

# Modelos de reward por braço (mesma lógica das células anteriores, sem class_weight)
mask_arm0_train = arm_train == 0
mask_arm1_train = arm_train == 1

model_arm0 = LogisticRegression(max_iter=1000, random_state=SEED)
model_arm0.fit(X_train[mask_arm0_train], y_train[mask_arm0_train])

model_arm1 = LogisticRegression(max_iter=1000, random_state=SEED)
model_arm1.fit(X_train[mask_arm1_train], y_train[mask_arm1_train])

prob_arm0_test = model_arm0.predict_proba(X_test)[:, 1]
prob_arm1_test = model_arm1.predict_proba(X_test)[:, 1]

# Direct Method: usa a própria probabilidade prevista pelos modelos como
# estimativa de recompensa esperada, cobrindo 100% do teste (em vez de só
# os casos que bateram com o histórico, como no replay da célula anterior)
dm_baseline = prob_arm0_test.mean()

resultados = []
for eps in [0.10, 0.05, 0.01, 0.00]:
    exploit_value = np.maximum(prob_arm0_test, prob_arm1_test)
    explore_value = (prob_arm0_test + prob_arm1_test) / 2
    dm_value = eps * explore_value + (1 - eps) * exploit_value
    dm_mean = dm_value.mean()
    lift = dm_mean - dm_baseline
    resultados.append({"epsilon": eps, "conversao_estimada": dm_mean, "lift_vs_baseline": lift})

tabela_final = pd.DataFrame(resultados)
print(f"Baseline (Direct Method): {dm_baseline:.4f}\n")
print(tabela_final.to_string(index=False))

/home/dev/miniconda3/envs/tc5/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Baseline (Direct Method): 0.1194

 epsilon  conversao_estimada  lift_vs_baseline
    0.10            0.125413          0.005999
    0.05            0.126570          0.007156
    0.01            0.127497          0.008083
    0.00            0.127728          0.008314


/home/dev/miniconda3/envs/tc5/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


a primeira avaliação (replay, célula anterior) mostrou o Epsilon-Greedy abaixo do baseline — mas isso era uma limitação do método de avaliação, não do modelo: o replay só confirma resultado nos ~54% dos casos em que a escolha do bandit bateu com o canal usado no histórico, e essa amostra parcial é enviesada, já que a atribuição histórica de canal não foi aleatória. Usando o Direct Method (a própria probabilidade prevista pelos modelos como estimativa de recompensa, cobrindo 100% do teste), o Epsilon-Greedy supera o baseline em todos os epsilons testados, com o lift crescendo conforme epsilon cai — o comportamento esperado, já que a política só troca de braço quando prevê um ganho real.